# ML-07 - Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/github/HimanshuSharma-2856/Flyrank-ml--internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)](https://colab.research.google.com/github/HimanshuSharma-2856/Flyrank-ml--internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook checks two rule signals, freezes one transparent baseline, writes a ranked queue, and reviews the top ten. All inputs are observed trailing-90-day fields from the starter CSV.

> No trend fields, labels, IDs, or future-window values are used as rule features.

## 1. Signal checks and one rule

**Rule in plain words:** flag pages that are both stale (at least 180 days since their last update) and visible (at least 1,000 impressions in the trailing 90 days). Rank flagged pages by their observed visibility, so a reviewer sees the highest-opportunity stale pages first.

The rule has one positive reason code, `stale_and_visible`, and one action label, `refresh_review`. Rows outside the rule remain in the output as `not_flagged` with action `monitor`, so the full ranking is reproducible.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

# Locate the repository root whether the notebook runs from the repo or work/notebooks.
_here = Path.cwd()
_candidates = [_here, *_here.parents, Path("/workspaces/Flyrank-ml-internship"), Path("/workspace")]
root_matches = [path for path in _candidates if (path / "data" / "raw" / "content_refresh_anonymized.csv").exists()]
if not root_matches:
    # VS Code's notebook kernel can start outside the virtual workspace on Windows.
    local_data_matches = Path.home().glob("Downloads/**/data/raw/content_refresh_anonymized.csv")
    root_matches = [data_path.parents[2] for data_path in local_data_matches]
if not root_matches:
    raise FileNotFoundError(f"Could not locate the repository data from notebook cwd: {_here}")
ROOT = root_matches[0]
DATA_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
OUTPUT_DIR = ROOT / "work" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(DATA_PATH)
print(f"Rows: {len(raw):,}")
print(f"Columns: {len(raw.columns)}")

# Signal 1: staleness is linked to the refresh flag idea.
raw["staleness_bucket"] = pd.cut(
    raw["days_since_last_update"],
    bins=[-1, 90, 180, 365, np.inf],
    labels=["0-90", "91-180", "181-365", "365+"],
)
staleness_table = (
    raw.groupby("staleness_bucket", observed=False)
    .agg(n=("content_id", "size"), median_impressions=("impressions_90d", "median"), median_clicks=("clicks_90d", "median"))
    .reset_index()
)
print("\nSignal 1 - staleness behind refresh flags")
print(staleness_table.to_string(index=False))
staleness_observed = staleness_table.dropna(subset=["median_impressions"]).reset_index(drop=True)
newest_visibility = staleness_observed["median_impressions"].iloc[0]
oldest_visibility = staleness_observed["median_impressions"].iloc[-1]
staleness_verdict = "CONFIRMED" if oldest_visibility >= newest_visibility else "MIXED"
print(f"Verdict: {staleness_verdict} - the oldest observed bucket has {'at least' if staleness_verdict == 'CONFIRMED' else 'less'} typical visibility than the newest observed bucket.")

# Signal 2: observed search volume is the volume/quick-win flag signal.
raw["volume_bucket"] = pd.cut(
    raw["impressions_90d"],
    bins=[-np.inf, 99, 999, 9999, np.inf],
    labels=["0-99", "100-999", "1k-9.9k", "10k+"],
)
volume_table = (
    raw.groupby("volume_bucket", observed=False)
    .agg(n=("content_id", "size"), median_clicks=("clicks_90d", "median"), median_ctr=("ctr", "median"))
    .reset_index()
)
print("\nSignal 2 - visible volume behind quick-win triage")
print(volume_table.to_string(index=False))
volume_verdict = "CONFIRMED" if volume_table["median_clicks"].is_monotonic_increasing else "MIXED"
print(f"Verdict: {volume_verdict} - higher-impression buckets {'show higher typical clicks.' if volume_verdict == 'CONFIRMED' else 'do not rise cleanly in typical clicks.'}")


Rows: 30,000
Columns: 44

Signal 1 - staleness behind refresh flags
staleness_bucket     n  median_impressions  median_clicks
            0-90 20655               472.0            1.0
          91-180  9171              1692.0            2.0
         181-365   169                16.0            0.0
            365+     5                 2.0            0.0
Verdict: MIXED - the oldest observed bucket has less typical visibility than the newest observed bucket.

Signal 2 - visible volume behind quick-win triage
volume_bucket    n  median_clicks  median_ctr
         0-99 7994            0.0        0.00
      100-999 8494            0.0        0.00
      1k-9.9k 9910            5.0        0.17
         10k+ 3602           50.0        0.23
Verdict: CONFIRMED - higher-impression buckets show higher typical clicks.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import json

# Freeze the transparent rule on observed, trailing-window fields only.
required_columns = {
    "content_id",
    "content_type",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
}
missing_columns = required_columns.difference(raw.columns)
assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"

stale_flag = raw["days_since_last_update"].ge(180)
visible_flag = raw["impressions_90d"].ge(1000)
raw["score"] = stale_flag.astype(int) * visible_flag.astype(int) * np.log1p(raw["impressions_90d"])
raw["reason_code"] = np.where(raw["score"].gt(0), "stale_and_visible", "not_flagged")
raw["action_label"] = np.where(raw["score"].gt(0), "refresh_review", "monitor")

queue_columns = [
    "content_id",
    "content_type",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "score",
    "reason_code",
    "action_label",
]
queue = (
    raw[queue_columns]
    .sort_values(["score", "impressions_90d", "content_id"], ascending=[False, False, True])
    .reset_index(drop=True)
)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))
queue["score"] = queue["score"].round(4)

csv_path = OUTPUT_DIR / "baseline_action_score.csv"
queue.to_csv(csv_path, index=False)

metrics = {
    "rows_scored": int(len(queue)),
    "flagged_rows": int(queue["reason_code"].eq("stale_and_visible").sum()),
    "flag_rate": float(queue["reason_code"].eq("stale_and_visible").mean()),
    "rule": "days_since_last_update >= 180 and impressions_90d >= 1000",
    "rank_metric": "log1p(impressions_90d) among flagged rows",
    "label_used": False,
}
(OUTPUT_DIR / "baseline_metrics.json").write_text(json.dumps(metrics, indent=2))
print(f"Wrote {csv_path}")
print(json.dumps(metrics, indent=2))
print("\nTop 10 queue preview:")
print(queue.head(10).to_string(index=False))

Wrote C:\Users\Himanshu\Downloads\flyrank-ml-internship-main\flyrank-ml-internship-main\work\outputs\baseline_action_score.csv
{
  "rows_scored": 30000,
  "flagged_rows": 12,
  "flag_rate": 0.0004,
  "rule": "days_since_last_update >= 180 and impressions_90d >= 1000",
  "rank_metric": "log1p(impressions_90d) among flagged rows",
  "label_used": false
}

Top 10 queue preview:
 rank           content_id    content_type  days_since_last_update  impressions_90d  clicks_90d  ctr  avg_position   score       reason_code   action_label
    1 content_cf56e2e2e282 keyword article                     194            61678          94 0.15          19.7 11.0297 stale_and_visible refresh_review
    2 content_7368877ea310 keyword article                     194            59472          77 0.13          24.8 10.9933 stale_and_visible refresh_review
    3 content_1bfaa38ff26c keyword article                     194            25715          60 0.23          22.2 10.1549 stale_and_visible refresh_revie

## 3. Top-10 review

*For each of the top 10: action, why it is there, and what would make it wrong.*

In [4]:
top10 = queue.head(10).copy()

# Human-readable review: every row gets an action, a measured reason, and a falsifier.
def wrong_if(row: pd.Series) -> str:
    if row["clicks_90d"] == 0:
        return "the impressions are unqualified visibility and the page has no observed clicks"
    if row["avg_position"] == 0:
        return "the position field is missing, so the apparent opportunity cannot be checked"
    if row["content_type"] == "feedly article":
        return "the page type has sparse keyword context and needs a content-specific review"
    return "the page was already refreshed, redirected, or intentionally left unchanged"

print("Top-10 review")
for _, row in top10.iterrows():
    action = row["action_label"]
    why = f"{int(row['days_since_last_update'])} days since update and {int(row['impressions_90d']):,} impressions"
    print(f"{int(row['rank']):02d}. {row['content_id']} - action={action}; why={why}; wrong_if={wrong_if(row)}")


Top-10 review
01. content_cf56e2e2e282 - action=refresh_review; why=194 days since update and 61,678 impressions; wrong_if=the page was already refreshed, redirected, or intentionally left unchanged
02. content_7368877ea310 - action=refresh_review; why=194 days since update and 59,472 impressions; wrong_if=the page was already refreshed, redirected, or intentionally left unchanged
03. content_1bfaa38ff26c - action=refresh_review; why=194 days since update and 25,715 impressions; wrong_if=the page was already refreshed, redirected, or intentionally left unchanged
04. content_0a91db491d14 - action=refresh_review; why=193 days since update and 13,299 impressions; wrong_if=the page was already refreshed, redirected, or intentionally left unchanged
05. content_5feee3994adb - action=refresh_review; why=194 days since update and 7,812 impressions; wrong_if=the page was already refreshed, redirected, or intentionally left unchanged
06. content_c2d929d83eaa - action=refresh_review; why=193 days

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
print("\nWeak-pick review")
weak_picks = top10[(top10["clicks_90d"] == 0) | (top10["avg_position"] == 0)]
if weak_picks.empty:
    print("No weak pick met the zero-click or missing-position test; inspect the top ten manually before trusting the rule.")
else:
    for _, row in weak_picks.iterrows():
        print(f"{int(row['rank']):02d}. {row['content_id']} - weak because clicks={row['clicks_90d']}, avg_position={row['avg_position']}")

# Leakage guard: these fields are forbidden because they are labels, label ingredients, IDs, or future-window data.
used_rule_features = {"days_since_last_update", "impressions_90d"}
forbidden_features = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
}
assert used_rule_features.isdisjoint(forbidden_features)
assert set(queue.columns).isdisjoint({"trend_direction", "trend_pct", "is_declining_label"})
assert queue["rank"].is_unique and queue["rank"].min() == 1
assert csv_path.exists()
print("\nLeakage check: PASS - rule uses only observed trailing-window fields; no labels or future-window fields.")
print("Queue check: PASS - ranks are unique and the CSV exists.")


Weak-pick review
07. content_b16bd7307b39 - weak because clicks=0, avg_position=31.0

Leakage check: PASS - rule uses only observed trailing-window fields; no labels or future-window fields.
Queue check: PASS - ranks are unique and the CSV exists.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.